# NeuroProfile on Colab

Runs `test_encode.py` (OOM gate) and `batch_encode.py` (corpus grind) on a Colab GPU instead of the 8 GB 4060.

**What Colab changes vs. the Arch box (only three things really matter):**
1. **VRAM.** A T4 is 16 GB (free), an L4/A100 more (Pro) — double+ the 4060. The `predict()` OOM should just disappear. Confirm from the `[vram after predict chunk N]` line in the smoke test.
2. **Downloads.** Do **not** download YouTube on Colab — its datacenter IPs are hard-blocked (worse than your flagged-IP case). Pre-download clips on the 4060 (where the winning yt-dlp recipe works), drop the `.mp4`s in Drive, and feed **file paths**, not URLs. This sidesteps the entire yt-dlp/deno/EJS/PO-token/cookies saga on the encode side.
3. **Persistence.** `/content` is scratch and dies on disconnect. Qdrant + timelines go to Drive (constraint #10). The scripts already take these as CLI args, so no code edit.

The Arch-specific pain is *gone*: the ctranslate2 execstack ELF-patch is unnecessary (stock Ubuntu kernel allows exec-stack), and the flaky-network Llama download is fast here.

## 0 · GPU check

In [1]:
import torch
try:
    from tribev2 import TribeModel
    import neuralset
    print(">>> IMPORT OK — tribev2 loads on torch", torch.__version__)
except Exception as e:
    import traceback; traceback.print_exc()
    print("\n>>> IMPORT FAILED:", type(e).__name__, "-", e)


>>> IMPORT FAILED: ModuleNotFoundError - No module named 'tribev2'


Traceback (most recent call last):
  File "/tmp/ipykernel_3141/152939229.py", line 3, in <cell line: 0>
    from tribev2 import TribeModel
ModuleNotFoundError: No module named 'tribev2'


In [1]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

name, memory.total [MiB], memory.free [MiB]
Tesla T4, 15360 MiB, 14913 MiB


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1 · Installs — run this, then **restart the runtime once**

All pip installs up front. After this cell: **Runtime ▸ Restart session**, then continue from Section 2. Restarting makes the pinned torch/numpy the versions that actually load (Colab preloads its own torch).

In [3]:
# numpy + torch must match tribev2's pins (numpy==2.2.6, torch>=2.5.1,<2.7).
# Installing the exact triple matches your handoff; if Colab's stock torch is already
# in [2.5.1, 2.7) you can skip the torch line and let tribev2 accept it.
!pip install -q numpy==2.2.6
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

# TRIBE v2 (not on PyPI). Its pins match ours, so it won't swap the torch triple.
!pip install -q "git+https://github.com/facebookresearch/tribev2.git"

# CPU-side pipeline deps (your repo's requirements, listed for a clean Colab env)
!pip install -q nibabel qdrant-client fastapi python-multipart uvicorn yt-dlp pytest scipy

# whisperX installed IN THIS env so TRIBE calls it directly (patched below) instead of
# uvx re-downloading ~3.5 GB every call. If this bumps torch, re-run the torch line above.
!pip install -q whisperx

print("installs done — now Runtime > Restart session, then run Section 2")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 80.9 MB/s eta 0:00:00:00:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.0 MB/s eta 0:00:0000:0100:01
ERROR: Could not find a version that satisfies the requirement torchvision==0.20.1 (from versions: 0.1.6, 0.2.0)
ERROR: No matching distribution found for torchvision==0.20.1
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 23.7 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.1/258.1 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.8/122.8 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.9/129.9 kB 1

### ⚠️ Restart the runtime now (Runtime ▸ Restart session), then run Section 2 onward.

## 2 · Drive, repo, auth, weights

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os
NP = '/content/drive/MyDrive/neuroprofile'   # durable root (survives disconnects)
for sub in ('qdrant_data', 'data/timelines', 'clips'):
    os.makedirs(f'{NP}/{sub}', exist_ok=True)
print('durable root:', NP)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
durable root: /content/drive/MyDrive/neuroprofile


In [5]:
# Get the repo onto FAST local scratch (/content), not Drive (Drive FUSE is slow for code).
# Make sure ica/ (frozen artifacts) and tests/reducer_reference.npz come along —
# reducer.py does np.load("ica/...") at import time and will crash without them.

# --- Option A: clone from GitHub (fill in your remote) ---
!git clone https://github.com/Mammbo/NeuroProfile.git /content/neuroprofile

# --- Option B: repo already in Drive — copy to scratch ---
# !cp -r /content/drive/MyDrive/neuroprofile/repo /content/neuroprofile

%cd /content/neuroprofile
!ls backend ica batch_encoding/

Cloning into '/content/neuroprofile'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 103 (delta 38), reused 98 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (103/103), 737.97 KiB | 2.56 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/content/neuroprofile
backend:
app.py	chunker.py  input_handler.py  reducer.py  stitcher.py  storage.py

batch_encoding/:
analyze_server.py  chunk_runner.py    setup_gpu.sh
batch_encode.py    _encode_worker.py  test_encode.py

ica:
atlas			     fsaverage5_glasser_labels.npy
fsaverage5_glasser_ids.json  region_system_map.json


In [6]:
import os, getpass
os.environ["HF_TOKEN"] = getpass.getpass("Paste HF read token: ").strip()

# verify it works:
from huggingface_hub import HfApi
print("auth OK as", HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"])

auth OK as Mammbo


In [7]:
# Colab's network is fast — this is the step that swung 47min->5.5hr on the 4060.
!hf download meta-llama/Llama-3.2-3B --include "*.safetensors" "config.json" "tokenizer*"
# older huggingface_hub: !huggingface-cli download meta-llama/Llama-3.2-3B --include "*.safetensors" "config.json" "tokenizer*"

Fetching 5 files:   0% 0/5 [00:00<?, ?it/s]Downloading 'config.json' to '/root/.cache/huggingface/hub/models--meta-llama--Llama-3.2-3B/blobs/47d4a5aa69cdef91a53b77f5c5583647a578ca0e.incomplete'

config.json: 100% 844/844 [00:00<00:00, 6.87MB/s]
Download complete. Moving file to /root/.cache/huggingface/hub/models--meta-llama--Llama-3.2-3B/blobs/47d4a5aa69cdef91a53b77f5c5583647a578ca0e
Fetching 5 files:  20% 1/5 [00:00<00:00,  5.29it/s]
tokenizer.json: 0.00B [00:00, ?B/s]

model-00002-of-00002.safetensors:   0% 0.00/1.46G [00:00<?, ?B/s]


tokenizer_config.json: 50.5kB [00:00, 104MB/s][A
Download complete. Moving file to /root/.cache/huggingface/hub/models--meta-llama--Llama-3.2-3B/blobs/cb9ec25536e44d86778b10509d3e5bdca459a5cf



model-00001-of-00002.safetensors:   0% 0.00/4.97G [00:00<?, ?B/s]
tokenizer.json: 557kB [00:00, 4.51MB/s]
tokenizer.json: 1.82MB [00:00, 8.85MB/s]
tokenizer.json: 9.09MB [00:00, 25.7MB/s]
Download complete. Moving file to /root/.cache/huggingface/hub/models--m

In [8]:
import os
os.environ["NLTK_ALLOW_PROXIED_URLOPEN"] = "1"
import nltk
for r in ["punkt_tab", "punkt"]:
    nltk.download(r)

# Patch TRIBE so get_events_dataframe calls whisperx directly instead of via `uvx`
# (uvx spins an isolated env and re-downloads ~3.5 GB every call -> looks frozen at 0%).
import tribev2, pathlib
et  = pathlib.Path(tribev2.__file__).parent / "eventstransforms.py"
src = et.read_text()
new = src.replace('["uvx", "whisperx"', '["whisperx"')
et.write_text(new)
print("uvx->whisperx patch:", "applied" if new != src else "NO CHANGE — check the pattern in eventstransforms.py")

# NOTE: no ctranslate2 execstack ELF-patch needed on Colab (Ubuntu kernel allows exec-stack).
# TRIBE's default whisperx is large-v3 fp16 on cuda. On a 16 GB T4 it may fit alongside TRIBE.
# If predict()+whisperx OOM even here, apply your 4060 edit (model="small", device="cpu",
# compute_type="int8") to eventstransforms.py.

AttributeError: 'numpy.ufunc' object has no attribute '__module__' and no __dict__ for setting new attributes

## 3 · Feed clips (upload `.mp4`s to Drive, then encode)

Upload your pre-downloaded clips to `MyDrive/neuroprofile/clips/`. The **file route** needs no yt-dlp, no cookies, no download — `resolve_source` sniffs magic bytes and goes straight to `chunk_video`.

In [9]:
!pip install -q ffmpeg-python

In [10]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/neuroprofile/clips', exist_ok=True)
!ls -la /content/drive/MyDrive/neuroprofile/clips

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 191461
-rw------- 1 root root  5468248 Aug 21 04:11 'Dog of Wisdom II [TnlakHr-O4w].mp4'
-rw------- 1 root root 28439109 Aug 21 05:48 'history of japan [Mh5LY4Mz15o].mp4'
-rw------- 1 root root 68840039 Aug 21 05:45 'history of the entire world, i guess [xuCn8ux2gbs].mp4'
-rw------- 1 root root  5458973 Aug 21 05:50 'How did the fall of Constantinople affect LeBron'\''s legacy？ [IBP5NUDP28A].mp4'
-rw------- 1 root root 16575426 Aug 21 05:46 'Prank interview with Elijah Wood [IfhMILe8C84].mp4'
-rw------- 1 root root 10546804 Aug 21 05:50 'Tame Impala - Loser (Official Video) [s3a4OQR-10M].mp4'
-rw------- 1 root root  2250717 Aug 21 04:06  test1.mp4
-rw------- 1 root root  3998826 Aug 21 05:54 'Victor Wembanyama blocks Lively then cooks him with nasty handles for 4-pt play 😭 [CkgubzZICHE].mp4'
-rw------- 1 root root 54475071 Aug 21 05:51 'Why no one guard

In [ ]:
%cd /content/neuroprofile
!python batch_encoding/test_encode.py /content/drive/MyDrive/neuroprofile/clips/test1.mp4

In [11]:
# Build a corpus of FILE PATHS (not URLs) and grind it, persisting to Drive.
import glob, pathlib
clips = sorted(glob.glob('/content/drive/MyDrive/neuroprofile/clips/*.mp4'))
pathlib.Path('/content/corpus.txt').write_text("\n".join(clips))
print(len(clips), "clips -> /content/corpus.txt")

9 clips -> /content/corpus.txt


In [ ]:
#TEST run with 2 clips 
%cd /content/neuroprofile
!python batch_encoding/batch_encode.py --corpus /content/corpus.txt --limit 2 \
    --qdrant-path   /content/drive/MyDrive/neuroprofile/qdrant_data \
    --timelines-dir /content/drive/MyDrive/neuroprofile/data/timelines


In [12]:
from qdrant_client import QdrantClient
QP = "/content/drive/MyDrive/neuroprofile/qdrant_data"
client = QdrantClient(path=QP)

print("collections:", [c.name for c in client.get_collections().collections])
print("count:", client.count("videos_v1").count)

pts, _ = client.scroll("videos_v1", limit=20, with_payload=True, with_vectors=False)
for p in pts:
    pl = p.payload
    print("\n—", pl["video_id"], "|", pl["title"], "| dur", pl.get("duration"), "s")
    print("   profile:", [round(x, 3) for x in pl["system_profile"]])
    print("   systems:", pl["system_names"])
    print("   moments:", len(pl.get("moments", [])), "| timeline:", pl["timeline_path"])

client.close()   # important — embedded Qdrant is single-process; close before anything else opens it

collections: ['videos_v1']
count: 7

— file:Victor Wembanyama blocks Lively then cooks him with nasty handles for 4-pt play 😭 [CkgubzZICHE] | Victor Wembanyama blocks Lively then cooks him with nasty handles for 4-pt play 😭 [CkgubzZICHE].mp4 | dur 48 s
   profile: [0.288, 0.151, 0.268, 0.099, 0.145, 0.032]
   systems: ['audiovisual_integration', 'social_sts_tpj', 'visual_motion', 'auditory', 'dmn_scene_medial_parietal', 'affect_reward']
   moments: 8 | timeline: 2ffd367759da.npz

— file:test1 | test1.mp4 | dur 61 s
   profile: [0.157, 0.147, 0.163, 0.091, 0.098, 0.043]
   systems: ['audiovisual_integration', 'social_sts_tpj', 'visual_motion', 'auditory', 'dmn_scene_medial_parietal', 'affect_reward']
   moments: 8 | timeline: 5f8d2090ce77.npz

— file:How did the fall of Constantinople affect LeBron's legacy？ [IBP5NUDP28A] | How did the fall of Constantinople affect LeBron's legacy？ [IBP5NUDP28A].mp4 | dur 220 s
   profile: [0.033, 0.016, 0.019, 0.06, 0.035, 0.026]
   systems: ['audiovis

In [19]:
!python batch_encoding/batch_encode.py --corpus /content/corpus.txt \
    --qdrant-path   /content/drive/MyDrive/neuroprofile/qdrant_data \
    --timelines-dir /content/drive/MyDrive/neuroprofile/data/timelines

# Persisted to Drive => a Colab disconnect mid-corpus is fine: re-run this cell and
# already-encoded ids are skipped (db.get_video != None -> status "skip"). 

Corpus: 9 source(s). Log -> /content/drive/MyDrive/neuroprofile/data/encode_log.jsonl
Mode: per-chunk subprocess isolation | audio+text=CPU, video=GPU

=== [1/9] /content/drive/MyDrive/neuroprofile/clips/Dog of Wisdom II [TnlakHr-O4w].mp4
[   skip] file:Dog of Wisdom II [TnlakHr-O4w]  Dog of Wisdom II [TnlakHr-O4w].mp4

=== [2/9] /content/drive/MyDrive/neuroprofile/clips/How did the fall of Constantinople aff
[   skip] file:How did the fall of Constantinople affect LeBron's legacy？ [IBP5NUDP28A]  How did the fall of Constantinople affect LeBron's legacy？ [

=== [3/9] /content/drive/MyDrive/neuroprofile/clips/Prank interview with Elijah Wood [IfhM
[   skip] file:Prank interview with Elijah Wood [IfhMILe8C84]  Prank interview with Elijah Wood [IfhMILe8C84].mp4

=== [4/9] /content/drive/MyDrive/neuroprofile/clips/Tame Impala - Loser (Official Video) [
[   skip] file:Tame Impala - Loser (Official Video) [s3a4OQR-10M]  Tame Impala - Loser (Official Video) [s3a4OQR-10M].mp4

=== [5/9] /conte

live inference

In [13]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
   -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
import os, subprocess, time
subprocess.Popen(["python","batch_encoding/analyze_server.py",
     "--qdrant-path","/content/drive/MyDrive/neuroprofile/qdrant_data",
     "--timelines-dir","/content/drive/MyDrive/neuroprofile/data/timelines",
     "--host","127.0.0.1","--port","8000"], cwd="/content/neuroprofile", env=dict(os.environ),
     stdout=open("/content/analyze.log","a"), stderr=subprocess.STDOUT)
time.sleep(8); print(open("/content/analyze.log").read()[-500:])
!cloudflared tunnel --url http://localhost:8000 

INFO:     Started server process [24828]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)

2026-08-26T16:18:30Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-26T16:18:30Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-26T16:18:34Z INF +--------------------------------------------------------------------------------------------+
20